# Limpieza de datos · Hackathon HORSE
### Línea 5 de montaje de motores · Valladolid

Este notebook limpia los datos de paradas **paso a paso**, explicando en cada parte qué se hace, por qué, y enseñándolo con gráficas.

**Datos de partida**
- `Datos/datos_hackaton.csv`: horas por operación y día en cada estado.
- `Datos/config_hackaton.csv`: las operaciones de la línea y su orden.

**Columnas de la tabla de datos**

| Columna | Significado |
|---|---|
| `TBF` | Horas **produciendo** (*Time Between Failures*) |
| `TAP` | Horas en **parada propia** (la máquina falla) |
| `DES` | Horas en **desabastecimiento** (no le llegan piezas de la anterior) |
| `SAT` | Horas en **saturación** (la siguiente no le acepta piezas) |
| `NAP` | **Número** de paradas propias |
| `NPR` | **Número de piezas** producidas |

**Supuesto clave: 22 h abiertas por día.** Son 3 turnos de 8 h (24 h) menos 2 h de descanso para comer en montaje. En un día perfecto, `TBF + TAP + DES + SAT = 22 h` para cada operación.

**Filosofía de la limpieza:** no se borra información. Las filas dudosas se **marcan** con *flags*, y una columna `incluir_analisis` indica qué entra al cálculo de cuellos de botella. Así cualquier decisión se puede revisar o deshacer.

## 0. Preparación

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

ENTRADA = Path("Datos")
SALIDA = Path("Datos_limpios")
SALIDA.mkdir(exist_ok=True)

T_ABIERTO = 22.0   # horas abiertas por día
TOL = 0.05         # tolerancia de redondeo (h)
ESTADOS = ["TBF", "TAP", "DES", "SAT"]
CLAVE = ["Year", "Week", "Jour", "Operacion"]

# Colores fijos para cada estado en todas las gráficas
COLORES = {"TBF": "#4C9F70", "TAP": "#D1495B", "DES": "#EDAE49", "SAT": "#00798C",
            "T_no_registrado": "#B0B0B0"}
NOMBRES = {"TBF": "Producción (TBF)", "TAP": "Parada propia (TAP)", "DES": "Desabastecimiento (DES)",
            "SAT": "Saturación (SAT)", "T_no_registrado": "No registrado"}
sns.set_theme(style="whitegrid", font_scale=0.95)
plt.rcParams["figure.dpi"] = 110

## 1. Carga y primer vistazo

Antes de tocar nada, miramos el tamaño, los tipos y unas filas de ejemplo.

In [ ]:
d = pd.read_csv(ENTRADA / "datos_hackaton.csv")
c = pd.read_csv(ENTRADA / "config_hackaton.csv")

print(f"Datos:  {d.shape[0]} filas x {d.shape[1]} columnas")
print(f"Config: {c.shape[0]} operaciones")
print(f"Semanas: {sorted(d.Week.unique())} | días por semana: {sorted(d.Jour.unique())}")
print(f"Operaciones distintas: {d.Operacion.nunique()}")
d.head()

In [ ]:
d[ESTADOS + ["NAP", "NPR"]].describe().round(2)

**Qué vemos:** 4 semanas × 5 días × 91 operaciones = 1820 filas. Una fila es *"cuántas horas pasó la operación X en cada estado el día D"*.

Ya hay pistas de suciedad en el `describe`: `NAP` llega a **23.022 paradas en un día** y `NPR` a 3314 piezas, cuando lo normal ronda las 1500.

## 2. Integridad básica

Comprobaciones mínimas antes de interpretar nada:
1. ¿Hay filas duplicadas (misma operación y mismo día dos veces)?
2. ¿Hay valores nulos o negativos?
3. ¿Coinciden las operaciones de la tabla de datos con las de la configuración?
4. ¿El orden de la configuración es único y continuo?

In [ ]:
print("Claves duplicadas:     ", d.duplicated(CLAVE).sum())
print("Valores nulos:         ", d.isna().sum().sum())
print("Valores negativos:     ", (d[ESTADOS + ["NAP", "NPR"]] < 0).sum().sum())
print("En datos y no en config:", sorted(set(d.Operacion) - set(c.Operacion)))
print("En config y no en datos:", sorted(set(c.Operacion) - set(d.Operacion)))
print("Orden_OP duplicado:    ", c.Orden_OP.duplicated().sum(), "| rango", c.Orden_OP.min(), "-", c.Orden_OP.max())

✅ La **estructura** está limpia: sin duplicados, nulos ni negativos, y datos y configuración tienen las mismas 91 operaciones.

⚠️ Los nombres son inconsistentes (`ST_0015`, `ST-0050`, `ST0261`), aunque coinciden exactamente entre los dos ficheros. Creamos un **nombre normalizado** solo para mostrarlo en gráficas, sin tocar la clave original.

La suciedad no está en la forma de la tabla sino **en los valores**. Vamos a buscarla.

In [ ]:
def normalizar_nombre(op):
    m = re.fullmatch(r"(ST|PM)[-_]?(\d+)(_\d+)?", op)
    return f"{m.group(1)}_{m.group(2)}{m.group(3) or ''}" if m else op

def tipo_operacion(op):
    if op.startswith("ST"): return "Estación (ST)"
    if op.startswith("PM"): return "Puesto manual (PM)"
    if op.startswith("Ma") or op.startswith("MAN"): return "Manutención / auxiliar"
    return "Otro"

c["Operacion_norm"] = c.Operacion.map(normalizar_nombre)
c["Tipo"] = c.Operacion.map(tipo_operacion)
print(c.Tipo.value_counts())
c[c.Operacion != c.Operacion_norm].head(8)

## 3. ¿Suman 22 h? Primera radiografía

Si los datos estuvieran bien, cada fila sumaría 22 h. Miramos la distribución de `TBF + TAP + DES + SAT`.

In [ ]:
d["T_total_raw"] = d[ESTADOS].sum(axis=1)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(d.T_total_raw, bins=np.arange(0, 27, 0.25), color="#5470C6", edgecolor="white")
ax.axvline(T_ABIERTO, color="black", ls="--", lw=1.5, label="22 h (tiempo abierto)")
ax.set(title="Horas registradas por fila (operación × día)", xlabel="TBF + TAP + DES + SAT (h)", ylabel="Nº de filas")
ax.legend(); plt.show()

print(f"Exactamente ~22 h: {((d.T_total_raw - 22).abs() <= TOL).sum()} filas")
print(f"Menos de 22 h:     {(d.T_total_raw < 22 - TOL).sum()} filas  (de ellas {(d.T_total_raw == 0).sum()} a cero)")
print(f"Más de 22 h:       {(d.T_total_raw > 22 + TOL).sum()} filas  (máximo {d.T_total_raw.max():.2f} h)")

**Tres problemas a la vista:**
- Un grupo de filas **a 0 h**: operaciones sin ningún dato.
- Muchas filas un poco **por debajo de 22 h**: falta tiempo sin estado asignado.
- Filas **por encima de 22 h**, hasta 26 h: imposible en un día de 22 h, así que hay estados solapados.

Los vamos resolviendo uno a uno. Antes hay algo más gordo que mirar: los días.

## 4. Días no productivos (línea cerrada)

Vemos cuántas piezas hizo la línea cada día y cuánta TAP media marcaron las máquinas.

In [ ]:
dia = (d.groupby(["Year", "Week", "Jour"])
         .agg(piezas_mediana=("NPR", "median"), tap_medio=("TAP", "mean"), tap_min=("TAP", "min"))
         .reset_index())
dia["etiqueta"] = "S" + dia.Week.astype(str) + "-D" + dia.Jour.astype(str)
dia["cerrado"] = (dia.piezas_mediana == 0) & (dia.tap_min >= T_ABIERTO - TOL)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
col = np.where(dia.cerrado, "#D1495B", "#4C9F70")
axes[0].bar(dia.etiqueta, dia.piezas_mediana, color=col)
axes[0].set(title="Piezas producidas por día (mediana de las operaciones)", ylabel="Piezas")
axes[1].bar(dia.etiqueta, dia.tap_medio, color=col)
axes[1].axhline(22, color="black", ls="--", lw=1)
axes[1].set(title="TAP media por operación (h)", ylabel="Horas")
axes[1].tick_params(axis="x", rotation=60)
from matplotlib.patches import Patch
axes[0].legend(handles=[Patch(color="#4C9F70", label="Día productivo"), Patch(color="#D1495B", label="Línea cerrada")])
plt.tight_layout(); plt.show()

dia[dia.cerrado][["etiqueta", "piezas_mediana", "tap_medio"]]

**Diagnóstico:** hay 4 días (S40-D1, S41-D5, S42-D1, S42-D5) en los que **las 91 operaciones** marcan 22 h de TAP y **0 piezas**.

No son 91 averías simultáneas: **la línea no trabajó** (festivo, fin de programa, mantenimiento planificado…). El sistema lo registró como parada propia.

**Decisión:** eliminar esos 4 días. Si se quedaran, toda la TAP saldría muy inflada (4 × 22 h por máquina) y ocultaría las paradas reales.

In [ ]:
antes = len(d)
d = d.merge(dia.loc[dia.cerrado, ["Year", "Week", "Jour"]], how="left", indicator=True)
d = d[d._merge == "left_only"].drop(columns="_merge").reset_index(drop=True)
print(f"Filas: {antes} -> {len(d)}  ({d[['Week','Jour']].drop_duplicates().shape[0]} días productivos)")

## 5. Radiografía por operación: ¿hay máquinas sin información útil?

Para cada operación calculamos cómo reparte de media sus 22 h entre los 4 estados, y cuántas piezas hace. Una operación sana produce la mayor parte del tiempo y hace unas 1500 piezas al día, como la línea.

In [ ]:
d = d.merge(c[["Operacion", "Orden_OP", "Operacion_norm", "Tipo"]], on="Operacion")
perfil = (d.groupby(["Orden_OP", "Operacion_norm", "Operacion"])[ESTADOS + ["NPR", "NAP"]]
            .mean().reset_index().sort_values("Orden_OP"))

fig, (ax, ax2) = plt.subplots(1, 2, figsize=(14, 18), gridspec_kw={"width_ratios": [3, 1]}, sharey=True)
izq = np.zeros(len(perfil))
y = np.arange(len(perfil))
for e in ESTADOS:
    ax.barh(y, perfil[e], left=izq, color=COLORES[e], label=NOMBRES[e], height=0.8)
    izq += perfil[e].values
ax.axvline(22, color="black", ls="--", lw=1)
ax.set_yticks(y, [f"{o:>2}. {n}" for o, n in zip(perfil.Orden_OP, perfil.Operacion_norm)], fontsize=7.5)
ax.invert_yaxis()
ax.set(title="Reparto medio de las horas del día por operación (en orden de línea)", xlabel="Horas")
ax.legend(loc="lower right", fontsize=8)
ax2.barh(y, perfil.NPR, color="#5470C6", height=0.8)
ax2.axvline(perfil.NPR.median(), color="black", ls="--", lw=1, label="Mediana línea")
ax2.set(title="Piezas / día", xlabel="Piezas"); ax2.legend(fontsize=8)
plt.tight_layout(); plt.show()

**Cómo leer la gráfica:** cada barra es una operación, en el orden de la línea. Verde = produciendo, rojo = parada propia, amarillo = sin piezas, azul = bloqueada.

Saltan a la vista **operaciones anómalas**:

| Operación | Qué se ve | Interpretación |
|---|---|---|
| `ST_0030` | Barra vacía, 0 piezas | Estación **sin datos**: inactiva o puenteada |
| `PM_2130` | Todo rojo (22 h TAP), 0 piezas | Está "parada" todo el mes **mientras la línea produce 1500 piezas/día**, así que en realidad no está en el flujo: **fuera de servicio** |
| `Ma_BM1_2`, `Ma_VES_0`, `Ma_VESRE` | Todo verde, 0 piezas, sin DES ni SAT | Contadores de manutención que siempre dicen "funcionando". **No aportan información** |
| `ManBM2_1` | Mucho rojo, 0 piezas, sin DES ni SAT | Señal errática de manutención |
| `ST2235` | Casi todo rojo pero con muchas piezas | **Contradicción**: lo vemos en el paso 7 |

**Decisión:** las 6 primeras se quedan en el fichero, pero **fuera del análisis** (`incluir_analisis = False`). Con ellas los métodos de cuello de botella darían resultados absurdos (por ejemplo, PM_2130 como "la máquina con más TAP").

In [ ]:
EXCLUIR = {
    "ST_0030": "Sin datos: todos los estados a 0 en todos los días productivos (inactiva/puenteada)",
    "PM2130": "TAP = 22 h todos los días y 0 piezas mientras la línea produce: fuera de servicio",
    "Ma_BM1_2": "Siempre TBF = 22 h, 0 piezas, sin DES/SAT: contador de manutención no informativo",
    "Ma_VES_0": "Siempre TBF = 22 h, 0 piezas, sin DES/SAT: contador de manutención no informativo",
    "Ma_VESRE": "Siempre TBF = 22 h, 0 piezas, sin DES/SAT: contador de manutención no informativo",
    "ManBM2_1": "0 piezas, sin DES/SAT y TAP errático (hasta 22 h con la línea produciendo)",
}
c["incluir_analisis"] = ~c.Operacion.isin(EXCLUIR)
c["motivo_exclusion"] = c.Operacion.map(EXCLUIR).fillna("")
d = d.merge(c[["Operacion", "incluir_analisis"]], on="Operacion")
print(f"Operaciones en el análisis: {c.incluir_analisis.sum()} de {len(c)}")
c.loc[~c.incluir_analisis, ["Orden_OP", "Operacion", "motivo_exclusion"]]

## 6. Cuadrar las horas a 22 h

Volvemos al problema del paso 3, ahora solo con días productivos.

### 6.1 Filas que suman **más** de 22 h

Un día solo tiene 22 h abiertas. Si una operación suma 24 h, algún estado se ha contado dos veces (solape). ¿Dónde pasa?

In [ ]:
d["T_total_raw"] = d[ESTADOS].sum(axis=1)
exceso = d.T_total_raw > T_ABIERTO + TOL
top = d[exceso].groupby("Operacion_norm").T_total_raw.agg(["count", "mean", "max"]).sort_values("count", ascending=False)

fig, ax = plt.subplots(figsize=(10, 4.5))
orden = top.index[:14]
sns.boxplot(data=d[d.Operacion_norm.isin(orden)], x="Operacion_norm", y="T_total_raw", order=orden, ax=ax, color="#9FB4E8")
ax.axhline(22, color="#D1495B", ls="--", lw=1.5, label="22 h")
ax.set(title="Operaciones que más veces superan 22 h", xlabel="", ylabel="Horas registradas / día")
ax.tick_params(axis="x", rotation=45); ax.legend(); plt.tight_layout(); plt.show()
print(f"{exceso.sum()} filas superan 22 h")
top.head(10).round(2)

El exceso es **sistemático** en unas pocas operaciones (PM_0090, PM_0110, ST_2135, ST_2376…): pasa casi todos los días y en la misma cantidad. No es ruido aleatorio.

Dos hipótesis:
- El sistema solapa estados (por ejemplo, cuenta a la vez SAT y TBF).
- Esas operaciones **no paran en el descanso de comida** y en realidad tienen 24 h. Es una pregunta para HORSE.

**Decisión (provisional):** reescalar proporcionalmente los 4 estados para que sumen 22 h. Así se mantiene el *reparto* entre estados, que es lo que usan los métodos de las flechas y del punto de inflexión. Los valores originales se guardan en columnas `*_raw`.

In [ ]:
for e in ESTADOS:
    d[f"{e}_raw"] = d[e]
d["flag_exceso_horas"] = exceso
factor = (T_ABIERTO / d.T_total_raw).where(exceso, 1.0)
d[ESTADOS] = d[ESTADOS].mul(factor, axis=0)
print("Máximo tras reescalar:", d[ESTADOS].sum(axis=1).max().round(3), "h")

### 6.2 Filas que suman **menos** de 22 h

Aquí falta tiempo que no está asignado a ningún estado: puede ser una pausa no registrada, una desconexión del sistema, un microcorte… **No sabemos qué es**, así que no lo inventamos.

**Decisión:** guardar la diferencia en una columna nueva, `T_no_registrado`. Así cada fila suma exactamente 22 h y el tiempo desconocido queda **visible**.

In [ ]:
d["T_no_registrado"] = (T_ABIERTO - d[ESTADOS].sum(axis=1)).clip(lower=0)
d["flag_defecto_horas"] = d.T_no_registrado > TOL

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.hist(d.T_total_raw, bins=np.arange(16, 27, 0.2), color="#9FB4E8", edgecolor="white")
a1.axvline(22, color="#D1495B", ls="--"); a1.set(title="ANTES: horas registradas", xlabel="h", ylabel="Filas")
a2.hist(d[ESTADOS + ["T_no_registrado"]].sum(axis=1), bins=np.arange(16, 27, 0.2), color="#4C9F70", edgecolor="white")
a2.axvline(22, color="#D1495B", ls="--"); a2.set(title="DESPUÉS: estados + no registrado", xlabel="h")
plt.tight_layout(); plt.show()

nr = d[d.flag_defecto_horas]
print(f"{len(nr)} filas con tiempo no registrado | media {nr.T_no_registrado.mean():.2f} h | máx {nr.T_no_registrado.max():.2f} h")

✅ Después del ajuste, **todas las filas suman 22 h**. La pila ya es comparable entre operaciones y entre días.

El tiempo no registrado es de **~1,2 h de media** en las filas afectadas. No es despreciable (~5 % del día), y conviene comentarlo en la presentación como **una limitación de los datos**.

## 7. Paradas propias sospechosas

Dos casos en los que la TAP **no se cree**:
1. **TAP de día completo** (≥ 22 h) en un día en que la línea sí produce. Si la línea está en serie y saca piezas, esa máquina no pudo estar parada todo el día.
2. **Número de paradas absurdo** (`NAP`). Lo vemos frente a la TAP.

In [ ]:
sub = d[d.incluir_analisis]
fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(sub.NAP + 1, sub.TAP_raw, s=14, alpha=0.5, color="#5470C6", label="Resto de operaciones")
s2235 = sub[sub.Operacion == "ST2235"]
ax.scatter(s2235.NAP + 1, s2235.TAP_raw, s=40, color="#D1495B", label="ST_2235")
ax.set_xscale("log")
ax.set(title="Nº de paradas propias vs horas de parada propia (por operación y día)",
       xlabel="NAP + 1 (escala logarítmica)", ylabel="TAP (h)")
ax.legend(); plt.tight_layout(); plt.show()

s2235[["Week", "Jour", "TBF_raw", "TAP_raw", "NAP", "NPR"]].head()

**ST_2235** está totalmente aislada: **~22.000 paradas al día** (una cada ~4 segundos) y ~15 h de TAP, pero hace **~2.400 piezas**, más que la propia línea.

Una máquina que pasa 15 de 22 h parada no puede producir más que la línea. Lo más probable es un **sensor que parpadea** y registra microcortes como paradas.

**Decisión:** marcarla con `flag_tap_sospechoso`. En el EDA habrá que decidir si se excluye o se trata aparte; si no, aparecería como "la máquina con más TAP" sin serlo.

In [ ]:
tap_dia = d.TAP_raw >= T_ABIERTO - TOL
nap_absurdo = d.NAP > 5000
d["flag_tap_sospechoso"] = tap_dia | nap_absurdo
print(f"TAP de día completo con la línea produciendo: {tap_dia.sum()} filas")
print(d[tap_dia].groupby("Operacion_norm").size().sort_values(ascending=False).to_string())
print(f"\nMás de 5000 paradas/día: {nap_absurdo.sum()} filas ({d[nap_absurdo].Operacion_norm.unique()})")

## 8. Contador de piezas (`NPR`)

Si la línea está en serie, **todas las operaciones deberían hacer más o menos las mismas piezas** (~1500/día). Las que se alejen mucho tienen un contador mal, o no están en serie.

In [ ]:
sub = d[d.incluir_analisis]
npr_op = sub.groupby(["Orden_OP", "Operacion_norm"]).NPR.median().reset_index()
mediana = npr_op.NPR.median()

fig, ax = plt.subplots(figsize=(14, 4.5))
colores = ["#D1495B" if v < 0.1 * mediana else "#EDAE49" if abs(v - mediana) > 0.3 * mediana else "#5470C6"
           for v in npr_op.NPR]
ax.bar(npr_op.Operacion_norm, npr_op.NPR, color=colores)
ax.axhline(mediana, color="black", ls="--", lw=1, label=f"Mediana línea ≈ {mediana:.0f}")
ax.set(title="Piezas/día por operación (mediana) — rojo: contador roto · naranja: muy distinto a la línea",
       ylabel="Piezas")
ax.tick_params(axis="x", rotation=90, labelsize=7); ax.legend(); plt.tight_layout(); plt.show()

| Operación | Qué pasa | Interpretación |
|---|---|---|
| `ST_0295` | Produce ~19 h/día pero cuenta **0-10 piezas** | **Contador roto**. Sus tiempos valen, sus piezas no |
| `ST_2376` y `ST_2375` | **~750** piezas cada una (la mitad) | Casi seguro **dos puestos en paralelo** que se reparten el trabajo |
| `ST_3300` | **2000-3300** piezas | Más que la línea: doble conteo, o cuenta subconjuntos |
| `MANOP749` | ~1000 piezas | Puesto manual auxiliar que no ve todas las piezas |

**Decisión:** marcar con `flag_npr_invalido` las filas en las que la máquina produce pero cuenta menos del 10 % de lo que hace la línea. Lo del paralelo y ST_3300 **no se corrige**: se documenta y se pregunta a HORSE, porque cambia cómo se aplica el método de las flechas.

In [ ]:
med_dia = d[d.incluir_analisis].groupby(["Week", "Jour"]).NPR.median().rename("npr_linea")
d = d.merge(med_dia, on=["Week", "Jour"], how="left")
d["flag_npr_invalido"] = d.incluir_analisis & (d.TBF > 1) & (d.NPR < 0.1 * d.npr_linea)
d = d.drop(columns="npr_linea")
print(f"Filas con NPR no fiable: {d.flag_npr_invalido.sum()}")
print(d[d.flag_npr_invalido].Operacion_norm.value_counts().to_string())

## 9. Columnas derivadas y resultado final

Añadimos lo que necesitará el análisis:
- `TAI = DES + SAT`: tiempo de parada **inducida**.
- Porcentajes de cada estado sobre las 22 h (`pct_*`).

In [ ]:
d["TAI"] = d.DES + d.SAT
for col in ESTADOS + ["TAI", "T_no_registrado"]:
    d[f"pct_{col}"] = 100 * d[col] / T_ABIERTO

columnas = (["Linea", "Year", "Week", "Jour", "Orden_OP", "Operacion", "Operacion_norm", "Tipo", "incluir_analisis"]
            + ESTADOS + ["TAI", "T_no_registrado", "NAP", "NPR"]
            + [f"pct_{x}" for x in ESTADOS + ["TAI", "T_no_registrado"]]
            + [f"{x}_raw" for x in ESTADOS] + ["T_total_raw", "flag_exceso_horas", "flag_defecto_horas",
                                               "flag_tap_sospechoso", "flag_npr_invalido"])
limpio = d[columnas].sort_values(["Week", "Jour", "Orden_OP"]).round(4).reset_index(drop=True)

# Validación final
suma = limpio[ESTADOS + ["T_no_registrado"]].sum(axis=1)
assert suma.between(T_ABIERTO - TOL, T_ABIERTO + TOL).all(), "Alguna fila no suma 22 h"
assert (limpio[ESTADOS] >= 0).all().all() and limpio.isna().sum().sum() == 0
print("✅ Validación OK: todas las filas suman 22 h, sin negativos ni nulos")
print(f"Filas: {len(limpio)} ({limpio[['Week','Jour']].drop_duplicates().shape[0]} días × {limpio.Operacion.nunique()} operaciones)")
print(f"Entran al análisis: {limpio.incluir_analisis.sum()} filas ({c.incluir_analisis.sum()} operaciones)")
limpio.head()

### Foto final de la línea limpia

El mismo gráfico de reparto de horas, ya con los datos limpios, solo con las operaciones válidas y **sumando horas de todos los días** antes de pasar a porcentaje.

In [ ]:
ok = limpio[limpio.incluir_analisis]
glob = ok.groupby(["Orden_OP", "Operacion_norm"])[ESTADOS + ["T_no_registrado"]].sum()
glob = (100 * glob.div(glob.sum(axis=1), axis=0)).reset_index()

fig, ax = plt.subplots(figsize=(15, 5))
abajo = np.zeros(len(glob))
for e in ["TAP", "SAT", "DES", "T_no_registrado", "TBF"]:
    ax.bar(glob.Operacion_norm, glob[e], bottom=abajo, color=COLORES[e], label=NOMBRES[e], width=0.85)
    abajo += glob[e].values
ax.set(title="Reparto del tiempo abierto por operación — 16 días productivos (datos limpios)", ylabel="% del tiempo")
ax.set_ylim(0, 100); ax.tick_params(axis="x", rotation=90, labelsize=7)
ax.legend(ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.32), fontsize=8)
plt.tight_layout(); plt.show()

Este gráfico es el **punto de partida del EDA**: ya se ve en qué tramos de la línea dominan la SAT y el DES. Es justo lo que usarán los métodos de detección de cuellos de botella.

## 10. Guardar resultados

In [ ]:
limpio.to_csv(SALIDA / "datos_limpios.csv", index=False)
c.to_csv(SALIDA / "config_limpia.csv", index=False)
print("Guardado en", SALIDA.resolve())

## Resumen de la limpieza

| # | Problema | Afecta a | Decisión |
|---|---|---|---|
| 4 | Días con la línea cerrada (todo TAP = 22 h, 0 piezas) | 4 días (364 filas) | **Eliminados** |
| 5 | Operaciones sin información útil | 6 operaciones | **Fuera del análisis** (se quedan marcadas) |
| 6.1 | Filas con más de 22 h (solapes) | 161 filas | **Reescaladas** a 22 h (originales en `*_raw`) |
| 6.2 | Filas con menos de 22 h | ~900 filas | Diferencia guardada en **`T_no_registrado`** |
| 7 | TAP imposible (ST_2235 con ~22.000 paradas/día; TAP de día completo con la línea en marcha) | 37 filas | **Marcadas** (`flag_tap_sospechoso`) |
| 8 | Contador de piezas roto (ST_0295) | 17 filas | **Marcadas** (`flag_npr_invalido`) |

**Resultado:** 16 días productivos × 85 operaciones válidas, con todas las filas cuadradas a 22 h.

### Preguntas para HORSE
1. **ST_2376 y ST_2375** hacen cada una la mitad de las piezas: ¿son puestos en **paralelo**?
2. Hay saltos en el orden de la config: `ST0261` antes de `ST0259`, `ST-2002` antes de `ST-2001` y `ST_0820-0850` después de `ST_2410`. ¿Es el flujo real?
3. **ST_3300** cuenta más piezas que la línea: ¿qué cuenta exactamente?
4. ¿Hay operaciones que **no paran en el descanso de comida** (24 h en vez de 22 h)?
5. Los **límites de la línea** (entrada y salida).